In [1]:
import pandas as pd

In [ ]:
prod_df = pd.read_csv("../../data/products_extended.csv")

In [55]:
def slugify_pandas(text: pd.Series) -> pd.Series:
    return text.str.lower().replace(" ", "-")

def map_ids(ser: pd.Series) -> pd.Series:
    return ser.astype("category").cat.codes + 1

def create_prod_hierarchy(df):
    
    df = df.copy()
    
    # Columns: category_id, subcategory_id, category_name, subcategory_name, category_slug, subcategory_slug

    df_hier = (
        df[["category", "subcategory"]]
        .drop_duplicates()
        .reset_index(drop=True)
        .rename(columns={"category": "category_name", "subcategory": "subcategory_name"})
    )
    df_hier["category_id"] = map_ids(df_hier["category_name"])
    df_hier["subcategory_id"] = map_ids(df_hier["subcategory_name"])
    df_hier["category_slug"] = slugify_pandas(df_hier["category_name"])
    df_hier["subcategory_slug"] = slugify_pandas(df_hier["subcategory_name"])

    return df_hier
    
    

In [56]:
prod_hier_df = create_prod_hierarchy(prod_df)

In [57]:
prod_hier_df

,category_name,subcategory_name,category_id,subcategory_id,category_slug,subcategory_slug
0,jewelry,bracelet,4,2,jewelry,bracelet
1,jewelry,earrings,4,4,jewelry,earrings
2,jewelry,necklace,4,5,jewelry,necklace
3,apparel,dress,2,3,apparel,dress
4,footwear,shoes,3,6,footwear,shoes
5,apparel,skirt,2,7,apparel,skirt
6,accessories,bag,1,1,accessories,bag
7,accessories,sunglasses,1,8,accessories,sunglasses
8,apparel,top blouse sweater,2,9,apparel,top blouse sweater


In [96]:
from sqlalchemy import create_engine
from sqlalchemy import text as sql_text

In [120]:
db = create_engine("sqlite:///ecom_new_prod_hier.db")

In [121]:
def execute_multi_sql(conn, sql_list):
    for sql in sql_list:
        conn.execute(sql_text(sql))
        conn.commit()

In [122]:
with db.connect() as conn:
    delete_sql_text = """
        DROP TABLE IF EXISTS product_hierarchy;
    """
    create_sql_text = """
        CREATE TABLE `product_hierarchy` (
            `id` INTEGER PRIMARY KEY AUTOINCREMENT,
            `category_id` INTEGER NOT NULL,
            `subcategory_id` INTEGER NOT NULL,
            `category_name` TEXT NOT NULL,
            `subcategory_name` TEXT NOT NULL,
            `category_slug` TEXT NOT NULL,
            `subcategory_slug` TEXT NOT NULL,
            UNIQUE (`category_id`, `subcategory_id`),
            FOREIGN KEY (`category_id`) REFERENCES `product_hierarchy`(`id`),
            FOREIGN KEY (`subcategory_id`) REFERENCES `product_hierarchy`(`id`)
        );
    """
    sql_list = [delete_sql_text, create_sql_text]
    execute_multi_sql(conn, sql_list)
    
    conn.commit()


In [129]:
with db.connect() as conn:
    delete_sql_text = "DROP TABLE product"
    create_sql_text = """
        CREATE TABLE IF NOT EXISTS `product` (
            `id` INTEGER PRIMARY KEY AUTOINCREMENT,
            `category_id` INTEGER NOT NULL,
            `subcategory_id` INTEGER NOT NULL,
            `name` TEXT NOT NULL,
            `slug` TEXT NOT NULL,
            `price` INTEGER NOT NULL,
            `description` TEXT NOT NULL,
            FOREIGN KEY (`category_id`) REFERENCES `product_hierarchy`(`category_id`),
            FOREIGN KEY (`subcategory_id`) REFERENCES `product_hierarchy`(`isubcategory_id`)
        );
    """
    sql_list = [delete_sql_text, create_sql_text]
    execute_multi_sql(conn, sql_list)


In [124]:
with db.connect() as conn:
    sql_list = [
        "delete from product_hierarchy",
        "UPDATE SQLITE_SEQUENCE SET seq = 0 WHERE name = 'product_hierarchy'"
    ]
    execute_multi_sql(conn, sql_list)
    prod_hier_df.to_sql("product_hierarchy", con=conn, if_exists="append", index=False)

In [125]:
def prepare_products_data(df_prod_raw, df_hier):

    df_prod = df_prod_raw.copy()
    df_prod.rename(columns={"category": "category_name", "subcategory": "subcategory_name"}, inplace=True)
    df_prod["category_slug"] = slugify_pandas(df_prod["category_name"])
    df_prod["subcategory_slug"] = slugify_pandas(df_prod["subcategory_name"])

    df_prod = pd.merge(
        df_prod,
        df_hier[["category_slug", "subcategory_slug", "category_id", "subcategory_id"]],
        on=["category_slug", "subcategory_slug"],
        how="left",
        validate="many_to_one"
    )
    df_prod["slug"] = slugify_pandas(df_prod["name"])

    cols_to_keep = [
        "name",
        "slug",
        "description",
        "category_id",
        "subcategory_id",
        "price",
    ]
    df_prod = df_prod[cols_to_keep]
    
    return df_prod

    

In [126]:
prod_df_prep = prepare_products_data(prod_df, prod_hier_df)

In [130]:
with db.connect() as conn:
    sql_list = [
        "delete from product",
        "UPDATE SQLITE_SEQUENCE SET seq = 0 WHERE name = 'product'"
    ]
    execute_multi_sql(conn, sql_list)
    prod_df_prep.to_sql("product", con=conn, if_exists="append", index=False)